In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 886.3/886.3 kB 21.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
Img_path = '/content/drive/MyDrive/Colab Notebooks/30DayCVChallenge/Day20_Animal_Pose/R.jpeg'

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 pose detection model
model = YOLO("yolov8n-pose.pt")  # Use an appropriate YOLOv8 pose model

# Perform inference on an image
results = model(Img_path)

# Access keypoints for each detected person
for result in results:  # Iterate through detections in results
    keypoints = result.keypoints  # Get keypoints of each person

    if keypoints is not None and hasattr(keypoints, 'data'):  # Check if keypoints and 'data' exist
        keypoints_data = keypoints.data.cpu().numpy()  # Move data to CPU and convert to numpy array

        for kp in keypoints_data[0]:  # Iterate through each keypoint
            x, y, confidence = kp[0], kp[1], kp[2]  # Access x, y, confidence
            print(f"Keypoint coordinates: ({x}, {y}), Confidence: {confidence}")
    else:
        print("No keypoints detected or 'data' attribute is missing.")

In [21]:
import cv2
import numpy as np
from ultralytics import YOLO

# Load YOLOv8 pose model
model = YOLO('yolov8n-pose.pt')  # Ensure the model is downloaded

# Define paths for input and output video
output_path = '/content/drive/MyDrive/Colab Notebooks/30DayCVChallenge/Day20_Animal_Pose/out_pushup5.mp4'
input_path = '/content/drive/MyDrive/Colab Notebooks/30DayCVChallenge/Day20_Animal_Pose/pushup4'

# Open video capture to get original properties
cap = cv2.VideoCapture(input_path)
fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Set up video writer with output file path, codec, frame rate, and resolution
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

# Initialize counters and states
pushup_count = 0
in_bottom_position = False

# Constants for keypoint indices
RIGHT_SHOULDER_INDEX = 6
RIGHT_ELBOW_INDEX = 8
RIGHT_WRIST_INDEX = 10
CONFIDENCE_THRESHOLD = 0.5  # Confidence threshold for keypoint filtering

def calculate_angle(a, b, c):
    """Calculate the angle between three points."""
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return angle if angle < 180.0 else 360 - angle

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLOv8 pose detection
    results = model(frame)

    # Draw pose detections on the frame
    annotated_frame = results[0].plot()  # YOLOv8 automatically plots pose keypoints

    # Extract keypoints for push-up logic
    keypoints = results[0].keypoints
    if keypoints is not None and hasattr(keypoints, 'data'):
        keypoints_data = keypoints.data.cpu().numpy()

        if len(keypoints_data[0]) > max(RIGHT_SHOULDER_INDEX, RIGHT_ELBOW_INDEX, RIGHT_WRIST_INDEX):
            try:
                # Extract specific keypoints (right shoulder, elbow, and wrist)
                right_shoulder = keypoints_data[0][RIGHT_SHOULDER_INDEX][:2]
                right_elbow = keypoints_data[0][RIGHT_ELBOW_INDEX][:2]
                right_wrist = keypoints_data[0][RIGHT_WRIST_INDEX][:2]

                # Calculate angle at the elbow
                angle = calculate_angle(right_shoulder, right_elbow, right_wrist)

                # Push-up detection logic
                if angle < 90 and not in_bottom_position:
                    in_bottom_position = True
                elif angle > 135 and in_bottom_position:
                    in_bottom_position = False
                    pushup_count += 1

                # Display angle and push-up count on the frame with increased text size
                text_angle = f"Angle: {int(angle)}"
                text_count = f"Push-Up Count: {pushup_count}"

                # Shadow effect for text (larger size)
                cv2.putText(annotated_frame, text_angle, (15, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 5)
                cv2.putText(annotated_frame, text_count, (15, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 0), 6)

                # Colorful text overlay with larger size
                cv2.putText(annotated_frame, text_angle, (10, 95), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (50, 150, 255), 4)
                cv2.putText(annotated_frame, text_count, (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 100, 100), 5)

            except IndexError:
                print("A required keypoint is missing in the detected pose.")
                continue

    # Write the annotated frame with overlay to the output video
    out.write(annotated_frame)

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Video saved to {output_path}")


0: 384x640 1 person, 294.8ms
Speed: 5.2ms preprocess, 294.8ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 278.1ms
Speed: 3.0ms preprocess, 278.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 262.3ms
Speed: 2.8ms preprocess, 262.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 271.5ms
Speed: 2.7ms preprocess, 271.5ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 231.2ms
Speed: 2.7ms preprocess, 231.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 235.3ms
Speed: 2.9ms preprocess, 235.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 232.0ms
Speed: 2.7ms preprocess, 232.0ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 219.8ms
Speed: 2.8ms preprocess, 219.8ms inference, 1.2ms postprocess per image at